In [1]:
# CELL 1: Setup, Environment Switch, and Path Configuration
import os
import sys
from pathlib import Path

os.environ["CACHE_LOOKBACK"] = "189"
os.environ["CACHE_START_DATE"] = "1998-01-01"


# ============================================================================
# --- ENVIRONMENT SWITCH ---
# Set to True if running in Google Colab.
# Set to False if running locally in Windows PC VSCode.
# ============================================================================
# 1. Detect environment
RUNNING_IN_COLAB = "google.colab" in sys.modules

if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    project_path = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks"
    )
    codebase_path = project_path / "notebooks_RLVR_v2"
else:
    project_path = Path("C:/Users/ping/Files_win10/python/py311/stocks")
    codebase_path = project_path / "notebooks_RLVR_v2"

data_path = project_path / "data" / "processed"
data_path.mkdir(parents=True, exist_ok=True)

output_path = codebase_path / "output"
output_path.mkdir(parents=True, exist_ok=True)

os.chdir(codebase_path)

In [9]:
import pickle

# Load the pickle file
with open(
    # r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\oos_evaluation_results.pkl",
    output_path / "oos_evaluation_results_v6.pkl",
    "rb",
) as f:
    results = pickle.load(f)

# Now you can use the data
print(results)

{'total_return': 0.7205508938635403, 'sharpe_ratio': 1.6931058777846673, 'equity_curve': [1.0, 1.0033353689744562, 1.0106658496607315, 1.0098690486862978, 1.0257582214156389, 1.0309682951939745, 1.0405034422794641, 1.057761529981961, 1.067747286255481, 1.0668895521203468, 1.0716562723057752, 1.074576367126592, 1.0641675326068758, 1.0547613399815277, 1.0560446275298927, 1.0556948945488545, 1.055618085018397, 1.055270926040036, 1.0593371042732493, 1.0529998203093975, 1.0466119639695388, 1.0443181572380191, 1.0460876046311478, 1.0378290517428697, 1.0317560717427507, 1.0289192403210017, 1.0305872252823136, 1.0355834187056592, 1.0367328162707246, 1.035790283084357, 1.0369104025880895, 1.0344490839748208, 1.0248354330368235, 1.0044128597144528, 0.9959463872770261, 0.9898592926785958, 0.9890440099180859, 0.9918410865805202, 0.9908608540543954, 0.986282121933464, 0.9871053189114203, 0.9835390366118626, 0.99008618899224, 0.9950772078124519, 0.9968744378848408, 1.0014518580590936, 1.016770451431

In [10]:
import numpy as np
import pandas as pd


def print_backtest_summary(results):
    print("=" * 65)
    print("                    RL BACKTEST EVALUATION SUMMARY")
    print("=" * 65)

    # 1. Print Core Metrics
    total_ret = results.get("total_return", 0.0)
    sharpe = results.get("sharpe_ratio", 0.0)
    steps = results.get("steps", 0)

    print(f"{'Total Cumulative Return:':<30} {total_ret * 100:.2f}%")
    print(f"{'Sharpe Ratio (Annual):':<30} {sharpe:.4f}")
    print(f"{'Total Trading Steps:':<30} {steps}")
    print("-" * 65)

    # 2. Print Array/List Previews
    for key in ["dates", "equity_curve"]:
        if key in results:
            lst = results[key]
            if len(lst) > 0:
                first = lst[0]
                last = lst[-1]
                # Format floating point curve values with precision
                first_str = (
                    f"{first:.4f}"
                    if isinstance(first, (float, np.floating))
                    else str(first)
                )
                last_str = (
                    f"{last:.4f}"
                    if isinstance(last, (float, np.floating))
                    else str(last)
                )
                print(
                    f"{f'{key.capitalize()} (len={len(lst)}):':<30} [{first_str} ... {last_str}]"
                )

    # 3. Print Nested Blotter Structure (Previewing the First Recorded Step)
    if "blotter" in results:
        blotter = results["blotter"]
        print("-" * 65)
        print(f"Blotter Logs: {len(blotter)} steps recorded.")
        if len(blotter) > 0:
            print("\n--- SAMPLE BLOTTER ENTRY (Step 1) ---")
            sample = blotter[0]
            for k, v in sample.items():
                if isinstance(v, np.ndarray):
                    preview = f"array(shape={v.shape}, dtype={v.dtype})"
                    if v.size > 4:
                        vals = ", ".join(
                            (
                                f"{x:.4f}"
                                if isinstance(x, (float, np.floating))
                                else str(x)
                            )
                            for x in v[:3]
                        )
                        preview += f" [{vals}, ...]"
                    else:
                        preview += f" {v.tolist()}"
                    print(f"  {k:<12} : {preview}")
                elif isinstance(v, list):
                    print(f"  {k:<12} : list(len={len(v)}) {v}")
                elif isinstance(v, (float, np.floating)):
                    print(f"  {k:<12} : {v:.6f}")
                else:
                    print(f"  {k:<12} : {v}")

    print("=" * 65)


# Example usage:
print_backtest_summary(results)

                    RL BACKTEST EVALUATION SUMMARY
Total Cumulative Return:       72.06%
Sharpe Ratio (Annual):         1.6931
Total Trading Steps:           843
-----------------------------------------------------------------
Dates (len=844):               [1677-09-21 00:12:43.145224193 ... 2026-05-27 00:00:00]
Equity_curve (len=844):        [1.0000 ... 1.7206]
-----------------------------------------------------------------
Blotter Logs: 843 steps recorded.

--- SAMPLE BLOTTER ENTRY (Step 1) ---
  decision_date : 2023-01-17
  buy_date     : 2023-01-18
  sell_date    : 2023-01-25
  universe_size : 885
  max_score    : -1.229877
  min_score    : -5.284475
  raw_actions  : list(len=13) [-0.03306369110941887, 0.041815806180238724, -0.0178590789437294, -0.04449110105633736, 0.055554311722517014, -0.0766587033867836, -0.14417365193367004, 0.016976896673440933, 0.1707310825586319, 0.024087507277727127, 0.18224872648715973, -0.10279399156570435, -0.0939982458949089]
  decoded_offset : 224


In [ ]:
import numpy as np


def print_all_blotter_steps(results_dict, limit=None):
    """
    Prints a clean, row-by-row overview of all blotter steps.
    Set 'limit' to an integer (e.g., 20) if you only want to inspect a subset.
    """
    blotter = results_dict.get("blotter", [])
    if not blotter:
        print("No blotter entries found.")
        return

    steps_to_print = blotter[:limit] if limit is not None else blotter

    print(f"\n--- PRINTING {len(steps_to_print)} BLOTTER STEPS ---")
    header = f"{'Step':<5} | {'Date':<10} | {'Reward':<10} | {'Tickers (Sample)':<45} | Action Preview"
    print(header)
    print("-" * len(header))

    for i, entry in enumerate(steps_to_print):
        step_num = i + 1

        # Extract and format values
        date_str = str(entry["date"]).split()[0]  # Extracts YYYY-MM-DD
        reward = entry.get("env_reward", 0.0)

        tickers = entry.get("tickers", [])
        tickers_str = ", ".join(tickers[:4])
        if len(tickers) > 4:
            tickers_str += "..."

        action = entry.get("action", np.array([]))
        action_preview = ", ".join(f"{float(x):.3f}" for x in action[:3])
        if len(action) > 3:
            action_preview += ", ..."

        print(
            f"{step_num:<5} | {date_str:<10} | {reward:<10.6f} | {tickers_str:<45} | [{action_preview}]"
        )


# Call the function (leave limit as None to print all 171 steps)
print_all_blotter_steps(results, limit=2)

In [ ]:
import pandas as pd
import IPython.display as ipydisplay

# 1. Configure Pandas to prevent column, row, and text truncation
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

# 2. Convert the blotter list of dicts to a DataFrame
df_blotter = pd.DataFrame(results["blotter"])

# 3. Clean and format columns (preserving all items)
df_display = df_blotter.copy()
df_display["env_reward"] = df_display["env_reward"].round(6)

# Convert all items in action to float rounded to 3 decimal places (without truncation)
df_display["action"] = df_display["action"].apply(
    lambda arr: [round(float(x), 3) for x in arr]
)

# Render the list of tickers in full
df_display["tickers"] = df_display["tickers"].apply(list)

# 4. Render the interactive table in VS Code
ipydisplay.display(df_display)
ipydisplay.display(df_display.iloc[0])

In [13]:
def verify_rlvr_pipeline(target_date_str, holding_period=5):
    target_date = pd.Timestamp(target_date_str)

    # =========================================================================
    # 1. TEMPORAL ALIGNMENT (No Lookahead Bias)
    # =========================================================================
    if target_date not in trading_calendar:
        print(f"[ERROR] {target_date_str} is not a valid trading day.")
        return

    date_idx = trading_calendar.get_loc(target_date)
    if date_idx + holding_period + 1 >= len(trading_calendar):
        print("[ERROR] Not enough future data to calculate the delayed reward.")
        return

    buy_date = trading_calendar[date_idx + 1]
    sell_date = trading_calendar[date_idx + 1 + holding_period]

    print(f"============================================================")
    print(f" RLVR PIPELINE VERIFICATION: {target_date_str}")
    print(f"============================================================")
    print(
        f"1. Decision Date: {target_date.strftime('%Y-%m-%d')} (Agent analyzes the market)"
    )
    print(
        f"2. Buy Date:      {buy_date.strftime('%Y-%m-%d')} (Shift +1 to execute orders)"
    )
    print(
        f"3. Sell Date:     {sell_date.strftime('%Y-%m-%d')} (Shift +{1+holding_period} to close positions)\n"
    )

    # =========================================================================
    # 2. OBSERVATION ADAPTER (33 Dimensions)
    # =========================================================================
    ensemble = feature_cube.xs(target_date, level="Date")
    macro_row = macro_df.loc[target_date]

    strat_mean = ensemble.mean(axis=0).fillna(0.0).values
    strat_std = ensemble.std(axis=0, ddof=0).fillna(0.0).values
    macro_vals = macro_row.fillna(0.0).values

    obs = np.concatenate(
        [np.asarray(strat_mean), np.asarray(strat_std), np.asarray(macro_vals)]
    ).astype(np.float32)
    obs = np.nan_to_num(obs, nan=0.0, posinf=0.0, neginf=0.0)

    feature_names = list(ensemble.columns)
    macro_names = list(macro_df.columns)

    obs_names = []
    obs_names.extend([f"1. Feature Mean: {name}" for name in feature_names])
    obs_names.extend([f"2. Feature StdDev: {name}" for name in feature_names])
    obs_names.extend([f"3. Macro: {name}" for name in macro_names])

    print("--- 33-DIMENSION OBSERVATION (The Market Weather) ---")
    for i, (name, val) in enumerate(zip(obs_names, obs)):
        print(f"Obs {i:02d} | {val:>9.4f} | {name}")

    # =========================================================================
    # 3. ABSOLUTE ZERO AGENT (Deterministic Inference)
    # =========================================================================
    # 1. Instantiate the skeleton
    agent = AbsoluteZeroAgent(obs_dim=33, action_dim=13).to(device)

    # 2. LOAD THE CHAMPION BRAIN (Update this path to your actual best .pt file)
    model_path = output_path / "model_checkpoints" / "champion_seed_420_ep_8_sh_1.43.pt"
    agent.load_state_dict(torch.load(model_path, map_location=device))

    # 3. Lock it into deterministic evaluation mode
    agent.eval()

    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        action_tensor = agent.actor_mean(obs_tensor)
    action_array = action_tensor.cpu().numpy()[0]

    feature_weights = action_array[:11]
    offset_raw = action_array[11]
    width_raw = action_array[12]

    print("\n--- 13-DIMENSION ACTION (The Agent's Strategy) ---")
    for i, (name, val) in enumerate(zip(feature_names, feature_weights)):
        direction = "POSITIVE (+)" if val > 0 else "NEGATIVE (-)"
        if val == 0:
            direction = "NEUTRAL  (0)"
        print(f"Act {i:02d} | {val:>7.4f} | {direction:<12} | Weight: {name}")

    print(f"Act 11 | {offset_raw:>7.4f} | SYSTEM DIAL  | Rank Offset")
    print(f"Act 12 | {width_raw:>7.4f} | SYSTEM DIAL  | Portfolio Width")

    # =========================================================================
    # 4. STRATEGY INTERPRETATION
    # =========================================================================
    print("\n--- AI STRATEGY INTERPRETATION ---")
    weights_series = pd.Series(feature_weights, index=feature_names)
    top_pos = weights_series.nlargest(2)
    top_neg = weights_series.nsmallest(2)

    print("Agent is HEAVILY FAVORING stocks with:")
    for name, val in top_pos.items():
        print(f"  -> High {name} (Weight: {val:.4f})")
    print("Agent is HEAVILY PENALIZING stocks with:")
    for name, val in top_neg.items():
        print(f"  -> High {name} (Weight: {val:.4f})")

    # =========================================================================
    # 5. SELECTION LOGIC (Matrix Routing & Integer Mapping)
    # =========================================================================
    #########################
    #########################
    # offset = int(np.interp(offset_raw, [-1, 1], [0, 50]))

    offset = int(np.interp(offset_raw, [-1, 1], [0, 500]))
    #########################
    #########################
    width = int(np.interp(width_raw, [-1, 1], [1, 10]))

    # Vectorized Math (No Bias is added in SelectionLogic)
    scores = pd.Series(ensemble.dot(feature_weights), index=ensemble.index)
    sorted_tickers = scores.sort_values(ascending=False)
    selected_tickers = sorted_tickers.index[offset : offset + width].tolist()

    print("\n--- RANKING LOGIC ---")
    print(f"1. Translated Offset : {offset}")
    print(f"2. Translated Width  : {width}")
    print(
        f"3. Result            : Buying Ranks {offset + 1} through {offset + width} from a universe of {len(sorted_tickers)} stocks."
    )

    # =========================================================================
    # 6. ALPHA LOGIC (Trade Execution & Reward Verification)
    # =========================================================================
    print("\n--- SELECTED TICKERS & TRADE VERIFICATION ---")
    total_pct_return = 0.0

    results = []
    for ticker in selected_tickers:
        buy_px = df_close.loc[buy_date, ticker]
        sell_px = df_close.loc[sell_date, ticker]

        if pd.isna(buy_px) or pd.isna(sell_px) or buy_px <= 0:
            print(f"{ticker:<5} | Skipped (Missing/Invalid Price Data)")
            continue

        pct_ret = (sell_px / buy_px) - 1.0
        total_pct_return += pct_ret

        results.append(
            {
                "Ticker": ticker,
                "Master Score": scores[ticker],
                "Buy Px": buy_px,
                "Sell Px": sell_px,
                "Pct Ret": pct_ret * 100,
            }
        )

    if len(results) > 0:
        res_df = pd.DataFrame(results)
        print(
            res_df.to_string(
                formatters={
                    "Master Score": "{:.2f}".format,
                    "Buy Px": "${:.2f}".format,
                    "Sell Px": "${:.2f}".format,
                    "Pct Ret": "{:.2f}%".format,
                }
            )
        )

        # AlphaLogic Verification
        arith_mean = total_pct_return / len(results)
        log_ret = float(np.log1p(arith_mean))

        print("\n--- ALPHA LOGIC REWARD MATH VERIFICATION ---")
        print(
            f"Arithmetic Mean Return :  {arith_mean * 100:.2f}% (The real-world portfolio ROI)"
        )
        print(
            f"Veritable Reward (LN)  :  {log_ret:.6f} (The exact number fed to the PPO Critic)"
        )
    else:
        print("No valid ticker prices found to execute trades on this date.")
    print("============================================================")

In [14]:
import pandas as pd

print("Loading DataFrames into RAM. Please wait...")

# cache_file_path = output_path / "alpha_cache_189d_2020.parquet"
cache_file_path = output_path / "alpha_cache_189d_1998.parquet"

feature_cube = pd.read_parquet(cache_file_path)
df_ohlcv = pd.read_parquet(project_path / "data" / "df_OHLCV_stocks_etfs.parquet")
macro_df = pd.read_parquet(data_path / "macro_df.parquet")

# 2. Re-derive the inputs needed for environment mapping
df_close = df_ohlcv["Adj Close"].unstack(level=0)
trading_calendar = pd.DatetimeIndex(df_close.dropna(how="all").index.sort_values())

Loading DataFrames into RAM. Please wait...


In [16]:
import pandas as pd
import numpy as np
import torch

from rl_discovery.agent import AbsoluteZeroAgent

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent = AbsoluteZeroAgent(obs_dim=33, action_dim=13).to(device)


# Try out the ultimate verification tool!
verify_rlvr_pipeline("2023-01-18", holding_period=5)

 RLVR PIPELINE VERIFICATION: 2023-01-18
1. Decision Date: 2023-01-18 (Agent analyzes the market)
2. Buy Date:      2023-01-19 (Shift +1 to execute orders)
3. Sell Date:     2023-01-26 (Shift +6 to close positions)

--- 33-DIMENSION OBSERVATION (The Market Weather) ---
Obs 00 |   -0.0807 | 1. Feature Mean: 189d_Log Price Gain
Obs 01 |    0.0009 | 1. Feature Mean: 189d_Sharpe (TRP)
Obs 02 |    0.0359 | 1. Feature Mean: 189d_Momentum (21d)
Obs 03 |    0.0507 | 1. Feature Mean: 189d_Info Ratio (63d)
Obs 04 |  -55.3717 | 1. Feature Mean: 189d_Oversold (-RSI)
Obs 05 |    0.0295 | 1. Feature Mean: 189d_Dip Buyer (-dd_21)
Obs 06 |    0.6323 | 1. Feature Mean: 189d_Range Position (20d)
Obs 07 |   -0.1905 | 1. Feature Mean: 189d_Return Autocorr (15d)
Obs 08 |   -0.0334 | 1. Feature Mean: 189d_Low Volatility (-ATRP)
Obs 09 |   -0.0000 | 1. Feature Mean: 189d_OBV Divergence (5d)
Obs 10 |   -0.0089 | 1. Feature Mean: 189d_Convexity
Obs 11 |    0.2612 | 2. Feature StdDev: 189d_Log Price Gain
Obs 12 